# 📊 EDA — Distribuição das Features
## Predictfy × Locaweb — FIAP Challenge 2026

Notebook exploratório: visualiza a distribuição das features geradas por `src/data/preprocessor.py`.

**Input:** `data/processed/incidents_features.parquet`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize": (14, 6),
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

df = pd.read_parquet("../data/processed/incidents_features.parquet")
print(f"Shape: {df.shape}")
print(f"Target — violações (1): {df['target_ola'].sum()} | OK (0): {(df['target_ola']==0).sum()}")
df.head()

## 1. Distribuição do Target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

vc = df["target_ola"].value_counts().sort_index()
axes[0].bar(["NAO (0)", "SIM (1)"], vc.values, color=["#5ac8fa", "#ff2d55"], edgecolor="white", width=0.5)
axes[0].set_title("Distribuição do Target — KPI Violado?", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Nº de incidentes")
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 80, f"{v:,}\n({v/len(df)*100:.2f}%)", ha="center", fontweight="bold")

axes[1].pie(vc.values, labels=["NAO (0)", "SIM (1)"],
            colors=["#5ac8fa", "#ff2d55"], autopct="%1.2f%%", startangle=90)
axes[1].set_title(f"Desbalanceamento — 1:{vc[0]//vc[1]}", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

## 2. Distribuição das Features Numéricas

In [ ]:
FEATURES = [c for c in df.columns if c != "target_ola"]

fig, axes = plt.subplots(5, 6, figsize=(22, 16))
axes = axes.flatten()

for i, col in enumerate(FEATURES):
    if i < len(axes):
        axes[i].hist(df[col], bins=30, color="#1a73e8", edgecolor="white", alpha=0.8)
        axes[i].set_title(col, fontsize=9)
        axes[i].tick_params(labelsize=7)

for j in range(len(FEATURES), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuição das Features Finais", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 3. Correlação das Features com Target

In [ ]:
corr = df.corr()["target_ola"].drop("target_ola").sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#ff2d55" if v > 0 else "#5ac8fa" for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor="white")
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Correlação de Pearson com target_ola", fontsize=12, fontweight="bold")
ax.set_xlabel("Correlação")
plt.tight_layout()
plt.show()